<a href="https://colab.research.google.com/github/nov-cpu/8730-project/blob/API_Pulling/API_Pulling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 28-07-2026 (Initial Version)

# !pip install google-search-results pandas openpyxl

  Preparing metadata (setup.py) ... done
  Created wheel for google-search-results: filename=google_search_results-2.4.2-py3-none-any.whl size=32010 sha256=3a8e561757ea1062e1ed92c386cb1015bafb47b410774a7dc9074d7efd224708
  Stored in directory: /root/.cache/pip/wheels/0c/47/f5/89b7e770ab2996baf8c910e7353d6391e373075a0ac213519e
Successfully built google-search-results


In [ ]:
# 28-07-2026 (Initial Version)

# import pandas as pd
# from serpapi import GoogleSearch
# import time
# import json

# # 1. Load the Data
# excel_path = '/content/alt_fuel_stations (Jul 26 2026).xlsx'

# # Read the Locations and Stations sheets
# df_locations = pd.read_excel(excel_path, sheet_name='Locations (Entity Table)')
# df_stations = pd.read_excel(excel_path, sheet_name='Stations (Parent Table)')

# # Clean up column names to avoid trailing whitespace issues
# df_locations.columns = df_locations.columns.str.strip()
# df_stations.columns = df_stations.columns.str.strip()

# # Merge the tables to associate the Station Name with its Address and Coordinates
# df_merged = pd.merge(
#     df_stations,
#     df_locations,
#     left_on='Location_ID (FK)',
#     right_on='Location_ID (PK)'
# )

# # 2. Setup SerpAPI Data Collection
# API_KEY = '4d175cb9ae871aa6a0c485c3512c0293d51453c4c26a4fe6f8dd1393ed92fb34'
# results_list = []

# # Note: We are using .head(5) to test the first 5 records and avoid burning API credits.
# # Remove .head(5) to run this across the entire dataset once you verify it works!
# for index, row in df_merged.head(5).iterrows():
#     station_name = row['Station_name']
#     street = row['street_Address']
#     city = row['City']
#     state = row['State']
#     lat = row['Latitude']
#     lon = row['Longitude']

#     # Construct a robust search query (e.g., "Ramada 1319 2nd St W Brooks AB")
#     search_query = f"{station_name} {street} {city} {state}"

#     params = {
#       "engine": "google_maps",
#       "q": search_query,
#       "ll": f"@{lat},{lon},15z", # Centers the map search around the exact lat/lon
#       "type": "search",
#       "api_key": API_KEY
#     }

#     try:
#         search = GoogleSearch(params)
#         results = search.get_dict()

#         place = None

#         # Check if an EXACT match exists (place_results)
#         if "place_results" in results:
#             place = results["place_results"]

#         # Fallback to checking if multiple local results exist
#         elif "local_results" in results and len(results["local_results"]) > 0:
#             place = results["local_results"][0]

#         if place:
#             # Extract the ratings, review count, and place information
#             extracted_data = {
#                 "Station_ID": row.get('Station_ID  (PK)'), # Using .get() is safer
#                 "Station_name": station_name,
#                 "Google_Place_ID": place.get("place_id"),
#                 "Google_Title": place.get("title"),
#                 "Rating": place.get("rating"),
#                 "Reviews_Count": place.get("reviews"),
#                 "Type": place.get("type"),
#                 "Address": place.get("address"),
#                 "Operating_Hours": place.get("operating_hours", {}).get("open_now")
#             }
#             results_list.append(extracted_data)
#             print(f"Successfully collected data for: {station_name}")
#         else:
#             print(f"No results found for: {search_query}")

#     except Exception as e:
#         print(f"Error fetching data for {search_query}: {e}")

#     # Respect API rate limits by pausing briefly between requests
#     time.sleep(1)

# # 3. Save the Extracted Data
# df_results = pd.DataFrame(results_list)

# # Export to JSON: Ideal for loading into MongoDB to handle unstructured/semi-structured data
# df_results.to_json('google_maps_station_ratings.json', orient='records', indent=4)

# # Export to CSV: Ideal for cleaning and loading into MySQL
# df_results.to_csv('google_maps_station_ratings.csv', index=False)

# print("\nData collection complete! JSON and CSV files have been saved.")

Successfully collected data for: Ramada
Successfully collected data for: Davis Chevrolet
Successfully collected data for: Gasonic Instruments
Successfully collected data for: International Motor Cars
Successfully collected data for: Residence Inn

Data collection complete! JSON and CSV files have been saved.


In [4]:
# 30-07-2026 (Version 2.0 - Individual Reviews Extraction)

!pip install google-search-results pandas openpyxl numpy

import pandas as pd
from serpapi import GoogleSearch
import time
import numpy as np

# ==========================================
# 1. HAVERSINE FORMULA (DISTANCE CALCULATION)
# ==========================================
def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6371.0 # Earth radius in kilometers
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

    return R * c # Distance in kilometers

# ==========================================
# 2. CONFIGURATION VARIABLES
# ==========================================
API_KEY = '4d175cb9ae871aa6a0c485c3512c0293d51453c4c26a4fe6f8dd1393ed92fb34'
EXCEL_PATH = '/content/alt_fuel_stations (Jul 26 2026).xlsx'

# Coordinates for Superstore Dougall Avenue Dock, Windsor, ON
USER_LAT = 42.2743
USER_LON = -83.0039
SEARCH_RADIUS_KM = 5.0
API_LIMIT = 5 # Scrape top 5 closest stations to protect API limits

# ==========================================
# 3. LOAD AND FILTER DATA BY PROXIMITY
# ==========================================
df_locations = pd.read_excel(EXCEL_PATH, sheet_name='Locations (Entity Table)')
df_stations = pd.read_excel(EXCEL_PATH, sheet_name='Stations (Parent Table)')

df_locations.columns = df_locations.columns.str.strip()
df_stations.columns = df_stations.columns.str.strip()

df_merged = pd.merge(
    df_stations,
    df_locations,
    left_on='Location_ID (FK)',
    right_on='Location_ID (PK)'
)

df_merged['Distance_KM'] = calculate_distance(
    USER_LAT, USER_LON,
    df_merged['Latitude'], df_merged['Longitude']
)

df_filtered = df_merged[df_merged['Distance_KM'] <= SEARCH_RADIUS_KM].copy()

total_nearby_count = len(df_filtered)
print(f"\n--- Output 1: Proximity Count ---")
print(f"There are {total_nearby_count} charging locations within {SEARCH_RADIUS_KM}km of the user.")
print(f"---------------------------------\n")

df_sorted = df_filtered.sort_values(by="Distance_KM")
df_top_closest = df_sorted.head(API_LIMIT)

print(f"--- Output 2: Scraping Details & Reviews for Top {API_LIMIT} Stations ---")

# ==========================================
# 4. SCRAPE MAPS & INDIVIDUAL REVIEWS
# ==========================================
results_list = []

for index, row in df_top_closest.iterrows():
    station_name = row['Station_name']
    street = row['street_Address']
    city = row['City']
    state = row['State']
    lat = row['Latitude']
    lon = row['Longitude']
    distance = round(row['Distance_KM'], 2)

    search_query = f"EV Charging Station {street} {city} {state}"

    params = {
      "engine": "google_maps",
      "q": search_query,
      "ll": f"@{lat},{lon},18z",
      "type": "search",
      "api_key": API_KEY
    }

    try:
        search = GoogleSearch(params)
        results = search.get_dict()

        place = None
        if "place_results" in results:
            place = results["place_results"]
        elif "local_results" in results and len(results["local_results"]) > 0:
            place = results["local_results"][0]

        data_id = place.get("data_id") if place else None
        user_reviews = []

        # SECONDARY API CALL: Scrape reviews text if data_id is present
        if data_id:
            try:
                review_params = {
                    "engine": "google_maps_reviews",
                    "data_id": data_id,
                    "api_key": API_KEY
                }
                review_search = GoogleSearch(review_params)
                review_results = review_search.get_dict()

                raw_reviews = review_results.get("reviews", [])
                for rev in raw_reviews:
                    user_reviews.append({
                        "Author": rev.get("user", {}).get("name", "Anonymous"),
                        "Rating": rev.get("rating"),
                        "Date": rev.get("date"),
                        "Comment": rev.get("snippet", rev.get("text", "No text provided"))
                    })
                print(f"  └─ Extracted {len(user_reviews)} individual reviews.")
            except Exception as rev_err:
                print(f"  └─ Error fetching reviews: {rev_err}")

        # Combine station details and extracted reviews
        extracted_data = {
            "Station_Name": station_name,
            "Distance_km": distance,
            "Address": place.get("address", f"{street}, {city}, {state}") if place else f"{street}, {city}, {state}",
            "Rating": place.get("rating", "N/A") if place else "N/A",
            "Reviews_Count": place.get("reviews", len(user_reviews)) if place else 0,
            "Operating_Hours": place.get("operating_hours", {}).get("open_now", "Unknown") if place else "Unknown",
            "Google_Maps_Link": f"https://www.google.com/maps?q={lat},{lon}",
            "Detailed_User_Reviews": user_reviews # Nested JSON structure ideal for MongoDB
        }

        results_list.append(extracted_data)
        print(f"Collected data: {station_name} ({distance} km away)")

    except Exception as e:
        print(f"Error fetching {search_query}: {e}")

    time.sleep(1)

# ==========================================
# 5. GENERATE OUTPUT FILES (CSV, JSON, HTML)
# ==========================================
df_results = pd.DataFrame(results_list)

if not df_results.empty:
    # 1. Export JSON: Contains full nested user review objects (MongoDB ready)
    df_results.to_json('nearby_station_ratings_with_reviews.json', orient='records', indent=4)

    # 2. Export CSV: Flattens reviews into text summaries for MySQL tabular storage
    df_csv = df_results.copy()
    df_csv['Detailed_User_Reviews'] = df_csv['Detailed_User_Reviews'].apply(
        lambda revs: " | ".join([f"[{r['Rating']}⭐] {r['Author']}: {r['Comment']}" for r in revs]) if revs else "No reviews"
    )
    df_csv.to_csv('nearby_station_ratings_with_reviews.csv', index=False)

    # 3. Export Interactive HTML Table
    df_html = df_results.copy()
    df_html['Google_Maps_Link'] = df_html['Google_Maps_Link'].apply(
        lambda x: f'<a href="{x}" target="_blank">View Pin on Maps</a>'
    )
    df_html['User_Reviews_Summary'] = df_html['Detailed_User_Reviews'].apply(
        lambda revs: "<br>".join([f"• <b>{r['Author']} ({r['Rating']}⭐):</b> {r['Comment']}" for r in revs[:3]]) if revs else "<i>No text reviews available</i>"
    )

    # Drop nested column for clean display table
    df_html_display = df_html[['Station_Name', 'Distance_km', 'Address', 'Rating', 'Reviews_Count', 'Operating_Hours', 'User_Reviews_Summary', 'Google_Maps_Link']]

    html_output = f"""
    <html>
    <head>
        <title>Closest {API_LIMIT} EV Charging Stations with Reviews</title>
        <style>
            body {{ font-family: Arial, sans-serif; margin: 20px; }}
            h2 {{ color: #2c3e50; }}
            table {{ border-collapse: collapse; width: 100%; margin-top: 20px; box-shadow: 0 2px 3px rgba(0,0,0,0.1); }}
            th, td {{ padding: 12px; text-align: left; border-bottom: 1px solid #ddd; vertical-align: top; }}
            th {{ background-color: #27ae60; color: white; }}
            tr:hover {{ background-color: #f5f5f5; }}
            a {{ color: #2980b9; text-decoration: none; font-weight: bold; }}
            a:hover {{ text-decoration: underline; }}
        </style>
    </head>
    <body>
        <h2>⚡ Top {API_LIMIT} Closest EV Charging Stations & User Reviews</h2>
        <p><strong>Base Location:</strong> Superstore Dougall Avenue Dock, Windsor, ON</p>
        {{table}}
    </body>
    </html>
    """

    final_html = html_output.replace('{{table}}', df_html_display.to_html(escape=False, index=False))

    with open("closest_stations_with_reviews.html", "w", encoding='utf-8') as f:
        f.write(final_html)

    print("\nSuccess! HTML, JSON (with nested reviews), and CSV files are ready in your Colab files panel.")
else:
    print("\nNo stations were found within the specified radius.")


--- Output 1: Proximity Count ---
There are 43 charging locations within 5.0km of the user.
---------------------------------

--- Output 2: Scraping Details & Reviews for Top 5 Stations ---
  └─ Extracted 8 individual reviews.
Collected data: CA DEVNSHR MALL STATION3 (0.19 km away)
  └─ Extracted 8 individual reviews.
Collected data: CA DEVNSHR MALL STATION 2 (0.19 km away)
  └─ Extracted 0 individual reviews.
Collected data: GM - Premier Chevrolet Buick GMC (0.5 km away)
  └─ Extracted 0 individual reviews.
Collected data: Premier Chevrolet Cadillac - Outside (0.54 km away)
  └─ Extracted 0 individual reviews.
Collected data: YNCU 650 DIVISION RD (0.69 km away)

Success! HTML, JSON (with nested reviews), and CSV files are ready in your Colab files panel.
